# Ariadne — Planning → Schematic Query Specialists

Drive the full **federated query workflow** end-to-end: the **planning agent** routes an analyst
question over the live graphs, and the two **source-specific query specialists** turn each routed
sub-question into a bounded, validated, read-only query and return **normalized evidence with provenance**.

- **Cypher specialist** — MITRE ATT&CK in Neo4j.
- **Gremlin specialist** — MITRE CAPEC in a Gremlin Server.

Both share one scaffold and differ only in four **swap points** (query builder, validator, connector,
normalizer). The discipline is *the model drafts, deterministic code grades*: a deterministic builder
proposes a query that the pre-run **gate** validates against the **source card** (the schematic) before
anything touches the database.

## Prerequisites
- **Graph stack running:** `uv run main.py --docker start` (ATT&CK in Neo4j, CAPEC in Gremlin).
- **Built artifacts + source cards:** `uv run main.py --download && uv run main.py --build all` then load the graphs.

## Launch (from the repo root)
```bash
uv run --with jupyterlab jupyter lab notebooks/planning-schematic-agents.ipynb
```
> One-shot CLI equivalent: `uv run main.py --query "<question>"`.

In [ ]:
# --- setup: make utils.* importable no matter where the notebook is launched ---
import sys, os, json, dataclasses
from pathlib import Path

_here = Path.cwd()
ROOT = next((p for p in [_here, *_here.parents] if (p / "pyproject.toml").exists()), _here)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.config import load_env
load_env()  # central .env: NEO4J_*/GREMLIN_*/SPECIALIST_* knobs; real env vars still win

from utils.planner import PlanningAgent, PlannerConfig, load_source_cards
from utils.planner.discovery import get_live_sources
from utils.planner.join_keys import resolve_join_key
from utils.planner.schema import Plan, SubQuestion, Limits, JoinDeclaration

from utils.specialists import build_specialist, SpecialistConfig, SpecialistTask
from utils.specialists.resolve import resolve
from utils.specialists.federation import SpecialistRouter
from utils.specialists.validation import QueryDraft
from utils.specialists.cypher.validator import CypherValidator
from utils.specialists.gremlin.validator import GremlinValidator

pcfg = PlannerConfig.from_env()
scfg = SpecialistConfig.from_env()
cards = load_source_cards(pcfg.data_dir)
print("repo root:", ROOT)
print("source cards:", sorted(cards))

## 1. Which graphs are live right now?

The planner content-probes each backend (connect → non-trivial node count → expected labels). The
specialists connect to the **same** env-driven endpoints (never the card's stale `connection.uri`).

In [ ]:
view = get_live_sources(cards, pcfg)
for ds in sorted(view.sources):
    s = view.sources[ds]
    flag = "LIVE" if s.live else "DOWN"
    detail = f"{s.node_count} nodes" if s.live else s.reason
    print(f"[{flag}] {ds:6s} ({s.backend:7s}) @ {s.endpoint}  —  {detail}")

## 2. The schematic — the source cards

Each **source card** is the specialist's grounding *and* its validation oracle: node labels,
relationship types, safety limits, and (for CAPEC) the allowed edge directions. The single declared
**canonical join key** is the ATT&CK technique id that bridges the two graphs.

In [ ]:
for ds, card in cards.items():
    print(f"=== {ds}  ({card.runtime_backend}) ===")
    print("  node_labels:      ", sorted(card.node_labels))
    print("  relationships:    ", sorted(card.relationship_types))
    print("  safety limits:    ", card.limits())
    if card.runtime_backend == "neo4j":
        print("  disallowed clauses:", card.safety.get("disallowed_clauses"))
    else:
        print("  disallowed steps: ", card.safety.get("disallowed_steps"))
        sample = dict(list(card.allowed_edge_patterns.items())[:4])
        print("  allowed_edge_patterns (sample):", sample)
    print()

key = resolve_join_key(cards["attack"], cards["capec"])
print("canonical join key:", key.name)
print(" ", key.description)

## 3. Plan the question

The planning agent routes the analyst question to the live sources, names the entities each
sub-question touches, draws the traversal limits from the card, and declares the cross-source join.
It *routes*; it never queries.

In [ ]:
QUESTION = (
    "Which CAPEC patterns related to credential theft map to ATT&CK techniques, "
    "and what mitigations appear for those techniques?"
)
agent = PlanningAgent(pcfg)
plan = agent.plan(QUESTION)

print("routed to:", sorted(plan.target_sources()))
for sq in plan.sub_questions:
    print(f"  [{sq.source}] {sq.id}: entities={sorted(sq.entities)}  "
          f"limits={sq.limits.to_dict()}  join_key={sq.join_key}")
print("joins:", [j.key for j in plan.joins])
for u in plan.unsupported:
    print("  ! unsupported:", u.description, "—", u.reason)

## 4. Inside the specialist loop — a helper

Each specialist runs **intake → resolve → draft → pre-run gate → execute → assess → normalize**.
Every checkpoint is recorded in an `EvaluatorTrace`. This helper prints that trace plus the
validated query, grounded summary, and the join values synthesis will key on.

In [ ]:
def show_result(result):
    print(f"TASK {result.task.sub_question_id} -> {result.task.source}   "
          f"(ok={result.ok}, partial={result.partial})")
    for t in result.traces:
        flag = "ok " if t.ok else "REJ"
        line = f"  [{t.step:9s}] {flag}"
        d = t.detail
        if t.step == "gate":
            line += f"  template={d.get('template')}  injected={d.get('injected')}"
        elif t.step == "execute":
            line += f"  rows={d.get('row_count')}  {d.get('timing_ms')}ms"
        elif t.step == "resolve":
            line += f"  labels={d.get('labels')}  seeds={d.get('seed_ids')}"
        elif t.step == "normalize":
            line += f"  records={d.get('record_count')}"
        if t.reasons:
            line += f"  reasons={t.reasons[:2]}"
        print(line)
    print("  QUERY  :", result.query)
    print("  SUMMARY:", result.summary)
    print("  JOINS  :", result.join_values()[:8])

## 5. Cypher specialist (ATT&CK / Neo4j) — step by step

A focused task with a concrete technique id. Watch the **resolve** step extract the `T1047` seed,
the **gate** inject a `LIMIT`, and the **normalize** step group the mitigations under the technique
with a clean `T1047` join value and full provenance.

In [ ]:
cyp = build_specialist("attack", cards["attack"], scfg)

attack_task = SpecialistTask(
    sub_question_id="demo-attack",
    source="attack",
    text="What mitigations apply to technique T1047?",
    entities=("Technique", "Mitigation"),
    limits=cards["attack"].limits(),
)

# The shared resolve step grounds the task against the card before any query is drafted:
r = resolve(attack_task, cards["attack"])
print("resolved -> labels:", r.labels, "| seed_ids:", r.seed_ids, "\n")

res_attack = cyp.run(attack_task)
show_result(res_attack)
print()
print("one normalized evidence record:")
print(json.dumps(res_attack.records[0].to_dict(), indent=2)[:900])

## 6. Gremlin specialist (CAPEC / Gremlin) — step by step

Two CAPEC tasks: the first follows `EXPLOITS_WEAKNESS` to the CWEs of CAPEC-66; the second walks the
**`MAPS_TO` ATT&CK bridge** (filtered to the `ATTACK` taxonomy) and emits **T-prefixed** join values —
the spine synthesis correlates on. Note the validator enforces edge direction and a single `.limit`.

In [ ]:
grm = build_specialist("capec", cards["capec"], scfg)

weak_task = SpecialistTask(
    sub_question_id="demo-capec-weak",
    source="capec",
    text="What weaknesses does CAPEC-66 exploit?",
    entities=("AttackPattern", "Weakness", "EXPLOITS_WEAKNESS"),
    limits=cards["capec"].limits(),
    seed_ids=("CAPEC-66",),
)
show_result(grm.run(weak_task))
print()

maps_task = SpecialistTask(
    sub_question_id="demo-capec-maps",
    source="capec",
    text="Which ATT&CK techniques does CAPEC-1 map to?",
    entities=("AttackPattern", "TaxonomyEntry", "MAPS_TO"),
    limits=cards["capec"].limits(),
    join_key="attack-technique-id",
    seed_ids=("CAPEC-1",),
)
show_result(grm.run(maps_task))

## 7. Federate the whole plan

`SpecialistRouter` dispatches every planned sub-question to the specialist serving its source and
bundles the normalized evidence — building a `join_index` over the shared key. The handoff is
**evidence, not prose**: the cross-source join and the final answer remain the synthesis agent's job.

In [ ]:
router = SpecialistRouter(scfg, cards=cards)
evidence = router.run_plan(plan)

for res in evidence.results:
    print(f"[{res.task.source}] {res.task.sub_question_id}: ok={res.ok} "
          f"records={len(res.records)}  ->  {res.summary}")
print("\npartial:", evidence.partial)
print("distinct ATT&CK join values surfaced by CAPEC (first 12):",
      sorted(evidence.join_index())[:12])

## 8. A real cross-source correlation

The planner routes; it does not *chain* (that is synthesis). To show the join lighting up, here is a
focused two-step plan: CAPEC-1 → its mapped ATT&CK technique, and that technique → its mitigations.
The same `T`-number then appears under **both** sources in the `join_index` — exactly the credential-
theft correlation the system exists to make auditable.

In [ ]:
focused = Plan(
    request="For CAPEC-1, which ATT&CK technique does it map to, and what mitigates that technique?",
    sub_questions=[
        SubQuestion("c1", "capec", "Which ATT&CK techniques does CAPEC-1 map to?",
                    ["AttackPattern", "MAPS_TO", "TaxonomyEntry"],
                    Limits(**cards["capec"].limits()), join_key="attack-technique-id"),
        SubQuestion("a1", "attack", "What mitigations apply to T1574.010?",
                    ["Technique", "Mitigation"],
                    Limits(**cards["attack"].limits()), join_key="attack-technique-id"),
    ],
    joins=[JoinDeclaration("attack-technique-id", ["capec", "attack"])],
)

fev = router.run_plan(focused)
for res in fev.results:
    show_result(res)
    print()

print("CROSS-SOURCE JOIN INDEX (a T-number reported by BOTH sources):")
for tval, entries in fev.join_index().items():
    srcs = sorted({e["source"] for e in entries})
    if len(srcs) > 1:
        print(f"  {tval}:")
        for e in entries:
            print(f"     - {e['source']:6s} {e['primary_id']}  {e['name']}")
print("\ncross_source_join_values:", fev.cross_source_join_values())

## 9. The gate keeps bad queries off the database

Both validators are pure deterministic Python and run **before** any DB call. They reject writes,
hallucinated labels/edges, wrong edge directions, and unbounded/over-deep traversals — the failure
modes that would otherwise silently corrupt the evidence or exhaust the graph.

In [ ]:
lim = {"max_traversal_depth": 3, "max_result_rows": 1000, "query_timeout_ms": 5000}
cv, gv = CypherValidator(), GremlinValidator()

print("CYPHER gate:")
for q in [
    "MATCH (t:Technique) WHERE t.attack_id IN $ids DELETE t RETURN t",   # write
    "MATCH (n:Foo) RETURN n.attack_id AS id",                            # unknown label
    "MATCH (g:Group)-[:USES*]->(t:Technique) RETURN t.name AS n",        # unbounded path
]:
    res = cv.validate(QueryDraft(query=q), cards["attack"], lim)
    print(f"  reject={not res.ok}  {q[:52]!r}\n     -> {res.reasons[:1]}")

print("\nGREMLIN gate:")
for q in [
    'g.V().hasLabel("AttackPattern").drop()',                                       # mutating
    'g.V().hasLabel("TaxonomyEntry").out("MAPS_TO").elementMap().limit(5)',         # wrong direction
    'g.V().hasLabel("AttackPattern").out("ChildOf").out("ChildOf").out("ChildOf").out("ChildOf").elementMap().limit(5)',  # over depth
]:
    res = gv.validate(QueryDraft(query=q), cards["capec"], lim)
    print(f"  reject={not res.ok}  {q[:58]!r}\n     -> {res.reasons[:1]}")

## Troubleshooting

- **A source shows DOWN** → start the stack: `uv run main.py --docker start`; for Neo4j ensure the
  `.env` `NEO4J_PASSWORD` matches the running container.
- **No source cards** → `uv run main.py --download && uv run main.py --build all`.
- **Tune via `.env`** → `SPECIALIST_QUERY_TIMEOUT_MS`, `SPECIALIST_MAX_RESULT_ROWS`,
  `SPECIALIST_MAX_RETRIES`, `NEO4J_URI`, `GREMLIN_URI`.
- **Full reference:** [`docs/specialist_agents.md`](../docs/specialist_agents.md) and
  [`docs/planner_agent.md`](../docs/planner_agent.md).